In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Column names
col_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

# Load data
train_df = pd.read_csv('data/raw/KDDTrain+.txt', names=col_names)
test_df = pd.read_csv('data/raw/KDDTest+.txt', names=col_names)

print(f"Training data: {train_df.shape}")
print(f"Test data: {test_df.shape}")
print("\n✅ Data loaded successfully!")

Training data: (125973, 43)
Test data: (22544, 43)

✅ Data loaded successfully!


In [2]:
# Check data types
print("DATA TYPES:")
print("=" * 50)
print(train_df.dtypes.value_counts())
print("\n")

# Separate columns by type
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"\nNumerical columns ({len(numerical_cols)}): {numerical_cols[:10]}... (and more)")

# Remove 'label' and 'difficulty' from numerical for now
if 'label' in numerical_cols: numerical_cols.remove('label')
if 'difficulty' in numerical_cols: numerical_cols.remove('difficulty')

DATA TYPES:
int64      24
float64    15
object      4
Name: count, dtype: int64


Categorical columns (4): ['protocol_type', 'service', 'flag', 'label']

Numerical columns (39): ['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised']... (and more)


In [3]:
# Check missing values
print("MISSING VALUES:")
print("=" * 50)
missing = train_df.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print("✅ No missing values found! Dataset is clean.")

MISSING VALUES:
✅ No missing values found! Dataset is clean.


In [4]:
# Create binary label for classification
train_df['binary_label'] = (train_df['label'] != 'normal').astype(int)
test_df['binary_label'] = (test_df['label'] != 'normal').astype(int)

print("Binary label distribution (Training):")
print(train_df['binary_label'].value_counts())
print(f"\n0 = Normal: {sum(train_df['binary_label']==0)} connections")
print(f"1 = Attack: {sum(train_df['binary_label']==1)} connections")

# Check test set
print("\n" + "="*50)
print("Binary label distribution (Test):")
print(test_df['binary_label'].value_counts())

Binary label distribution (Training):
binary_label
0    67343
1    58630
Name: count, dtype: int64

0 = Normal: 67343 connections
1 = Attack: 58630 connections

Binary label distribution (Test):
binary_label
1    12833
0     9711
Name: count, dtype: int64


In [5]:
# For binary classification (attack vs normal)
# Features: all columns except 'label' and 'binary_label' and 'difficulty'
feature_cols = [col for col in train_df.columns if col not in ['label', 'binary_label', 'difficulty']]

X_train = train_df[feature_cols]
y_train_binary = train_df['binary_label']

X_test = test_df[feature_cols]
y_test_binary = test_df['binary_label']

print(f"Training features shape: {X_train.shape}")
print(f"Training labels shape: {y_train_binary.shape}")
print(f"\nTest features shape: {X_test.shape}")
print(f"Test labels shape: {y_test_binary.shape}")

Training features shape: (125973, 41)
Training labels shape: (125973,)

Test features shape: (22544, 41)
Test labels shape: (22544,)


In [6]:
# Identify categorical columns in features
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical features to encode: {cat_cols}")

# Create label encoders for each categorical column
encoders = {}
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# Encode each categorical column
for col in cat_cols:
    le = LabelEncoder()
    # Fit on training data only
    X_train_encoded[col] = le.fit_transform(X_train[col].astype(str))
    
    # Transform test data (handle unseen categories)
    # For test, map unseen values to -1 or most frequent
    X_test_encoded[col] = X_test[col].astype(str).map(
        lambda s: le.transform([s])[0] if s in le.classes_ else -1
    )
    encoders[col] = le
    
    print(f"✅ Encoded '{col}': {len(le.classes_)} unique values")

print("\n✅ Categorical encoding complete!")
print(f"Training data shape after encoding: {X_train_encoded.shape}")

Categorical features to encode: ['protocol_type', 'service', 'flag']
✅ Encoded 'protocol_type': 3 unique values
✅ Encoded 'service': 70 unique values
✅ Encoded 'flag': 11 unique values

✅ Categorical encoding complete!
Training data shape after encoding: (125973, 41)


In [7]:
# Compare before and after encoding
print("BEFORE vs AFTER ENCODING:")
print("=" * 50)
for col in cat_cols[:3]:  # Show first 3 categorical columns
    print(f"\n{col}:")
    print(f"  Original (first 5): {X_train[col].head().tolist()}")
    print(f"  Encoded (first 5):  {X_train_encoded[col].head().tolist()}")

BEFORE vs AFTER ENCODING:

protocol_type:
  Original (first 5): ['tcp', 'udp', 'tcp', 'tcp', 'tcp']
  Encoded (first 5):  [1, 2, 1, 1, 1]

service:
  Original (first 5): ['ftp_data', 'other', 'private', 'http', 'http']
  Encoded (first 5):  [20, 44, 49, 24, 24]

flag:
  Original (first 5): ['SF', 'SF', 'S0', 'SF', 'SF']
  Encoded (first 5):  [9, 9, 5, 9, 9]


In [8]:
# Get numerical columns (excluding encoded categoricals)
num_cols = X_train_encoded.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Numerical features to scale ({len(num_cols)}): {num_cols[:10]}...")

# Initialize scaler
scaler = StandardScaler()  # or MinMaxScaler()

# Fit on training data and transform both train and test
X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train_encoded[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test_encoded[num_cols])

print("\n✅ Feature scaling complete!")

# Check before/after scaling
print("\nBEFORE vs AFTER SCALING (first numerical column):")
col = num_cols[0]
print(f"Column: {col}")
print(f"  Before - Min: {X_train_encoded[col].min():.2f}, Max: {X_train_encoded[col].max():.2f}, Mean: {X_train_encoded[col].mean():.2f}")
print(f"  After  - Min: {X_train_scaled[col].min():.2f}, Max: {X_train_scaled[col].max():.2f}, Mean: {X_train_scaled[col].mean():.2f}")

Numerical features to scale (41): ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot']...

✅ Feature scaling complete!

BEFORE vs AFTER SCALING (first numerical column):
Column: duration
  Before - Min: 0.00, Max: 42908.00, Mean: 287.14
  After  - Min: -0.11, Max: 16.36, Mean: 0.00


In [9]:
print("✅✅✅ DATA PREPROCESSING COMPLETE! ✅✅✅")
print("=" * 60)
print(f"Final training data shape: {X_train_scaled.shape}")
print(f"Final test data shape: {X_test_scaled.shape}")
print("\nData is now ready for machine learning!")
print("✓ Categorical features encoded")
print("✓ Numerical features scaled")
print("✓ Binary labels created")
print("\nFirst 5 rows of preprocessed data:")
X_train_scaled.head()

✅✅✅ DATA PREPROCESSING COMPLETE! ✅✅✅
Final training data shape: (125973, 41)
Final test data shape: (22544, 41)

Data is now ready for machine learning!
✓ Categorical features encoded
✓ Numerical features scaled
✓ Binary labels created

First 5 rows of preprocessed data:


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
0,-0.110249,-0.124706,-0.686785,0.751111,-0.007679,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,-0.324063,-0.818890,-0.782367,-0.280282,0.069972,-0.289103,-0.639532,-0.624871,-0.224532,-0.376387
1,-0.110249,2.219312,0.781428,0.751111,-0.007737,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,0.734343,-1.035688,-1.161030,2.736852,2.367737,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387
2,-0.110249,-0.124706,1.087305,-0.736235,-0.007762,-0.004919,-0.014089,-0.089486,-0.007736,-0.095076,...,0.734343,-0.809857,-0.938287,-0.174417,-0.480197,-0.289103,1.608759,1.618955,-0.387635,-0.376387
3,-0.110249,-0.124706,-0.442083,0.751111,-0.007723,-0.002891,-0.014089,-0.089486,-0.007736,-0.095076,...,-1.533670,1.258754,1.066401,-0.439078,-0.383108,0.066252,-0.572083,-0.602433,-0.387635,-0.345084
4,-0.110249,-0.124706,-0.442083,0.751111,-0.007728,-0.004814,-0.014089,-0.089486,-0.007736,-0.095076,...,0.734343,1.258754,1.066401,-0.439078,-0.480197,-0.289103,-0.639532,-0.624871,-0.387635,-0.376387


In [11]:
# Save preprocessed data for later use
import joblib

# Save to processed folder
X_train_scaled.to_csv('data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('data/processed/X_test.csv', index=False)
pd.DataFrame(y_train_binary).to_csv('data/processed/y_train.csv', index=False)
pd.DataFrame(y_test_binary).to_csv('data/processed/y_test.csv', index=False)

# Save encoders and scaler for future use
joblib.dump(encoders, 'models/encoders.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print("✅ Preprocessed data saved to 'data/processed/' folder")
print("✅ Encoders and scaler saved to 'models/' folder")

✅ Preprocessed data saved to 'data/processed/' folder
✅ Encoders and scaler saved to 'models/' folder
